# Bedrock Nova 图像和视频内容审核

本notebook使用Amazon Bedrock Nova模型进行图像和视频的内容审核，重点检测涉黄涉暴内容。

## 特点
- 🖼️ **图像审核**: 使用Nova模型分析图像内容
- 🎬 **视频审核**: 使用Nova模型分析视频内容
- ⏱️ **延迟统计**: 精确测量API响应时间
- 🔍 **统一标准**: 图像和视频使用相同的AI模型
- 📊 **详细报告**: JSON格式的结构化输出

## 1. 环境设置和导入

In [1]:
import base64
import boto3
import json
import time
import os
from datetime import datetime
from typing import Dict, List

# 创建Bedrock客户端
bedrock_client = boto3.client(
    "bedrock-runtime",
    region_name="us-west-2",
)

MODEL_ID = "us.amazon.nova-lite-v1:0"

print("✅ Bedrock客户端初始化完成")
print(f"✅ 模型: {MODEL_ID}")
print(f"✅ 区域: us-west-2")

✅ Bedrock客户端初始化完成
✅ 模型: us.amazon.nova-lite-v1:0
✅ 区域: us-west-2


## 2. 图像内容审核功能

In [2]:
def analyze_image_content(image_path: str, custom_prompt: str = None) -> Dict:
    """
    使用Bedrock Nova模型分析图像内容（涉黄涉暴审核）
    
    Args:
        image_path: 图像文件路径
        custom_prompt: 自定义提示词
    
    Returns:
        包含分析结果和时延信息的字典
    """
    start_time = time.time()
    
    try:
        # 读取图像文件并编码为Base64
        with open(image_path, "rb") as image_file:
            binary_data = image_file.read()
            base_64_encoded_data = base64.b64encode(binary_data)
            base64_string = base_64_encoded_data.decode("utf-8")
        
        # 默认审核提示词
        if custom_prompt is None:
            custom_prompt = """请仔细分析这张图像，并以JSON格式返回以下信息（使用中文回答）：
{
  "description": "图像的详细描述",
  "contains_violence": "是否包含暴力内容（是/否）",
  "violence_details": "如果包含暴力内容，请详细说明",
  "contains_porn": "是否包含色情内容（是/否）",
  "porn_details": "如果包含色情内容，请详细说明",
  "safety_rating": "安全等级（安全/警告/危险）",
  "confidence_score": "判断的置信度（1-10分）",
  "age_rating": "适合年龄段"
}"""
        
        # 定义系统提示
        system_list = [
            {
                "text": "你是一个专业的图像内容审核专家，擅长识别图像中的暴力和色情内容。请客观、准确地分析图像内容。"
            }
        ]
        
        # 定义用户消息
        message_list = [
            {
                "role": "user",
                "content": [
                    {
                        "image": {
                            "format": "png",
                            "source": {"bytes": base64_string},
                        }
                    },
                    {
                        "text": custom_prompt
                    },
                ],
            }
        ]
        
        # 配置推理参数
        inf_params = {"maxTokens": 400, "topP": 0.1, "topK": 20, "temperature": 0.2}
        
        native_request = {
            "schemaVersion": "messages-v1",
            "messages": message_list,
            "system": system_list,
            "inferenceConfig": inf_params,
        }
        
        # 调用Bedrock API
        api_start_time = time.time()
        response = bedrock_client.invoke_model(modelId=MODEL_ID, body=json.dumps(native_request))
        model_response = json.loads(response["body"].read())
        api_end_time = time.time()
        
        # 计算时延
        total_latency = api_end_time - start_time
        api_latency = api_end_time - api_start_time
        
        # 提取响应内容
        content_text = model_response["output"]["message"]["content"][0]["text"]
        
        # 尝试解析JSON响应
        try:
            parsed_content = json.loads(content_text)
        except json.JSONDecodeError:
            # 如果不是有效JSON，创建结构化响应
            parsed_content = {
                "description": content_text,
                "contains_violence": "未知",
                "contains_porn": "未知",
                "safety_rating": "未知",
                "raw_response": content_text
            }
        
        return {
            'success': True,
            'file_path': image_path,
            'file_type': 'image',
            'analysis_result': parsed_content,
            'raw_response': content_text,
            'model_id': MODEL_ID,
            'usage': model_response.get('usage', {}),
            'latency': {
                'total_ms': round(total_latency * 1000, 2),
                'api_ms': round(api_latency * 1000, 2)
            },
            'timestamp': datetime.now().isoformat()
        }
        
    except Exception as e:
        return {
            'success': False,
            'file_path': image_path,
            'file_type': 'image',
            'error': str(e),
            'latency': {
                'total_ms': round((time.time() - start_time) * 1000, 2),
                'api_ms': 0
            },
            'timestamp': datetime.now().isoformat()
        }

print("✅ 图像审核功能已定义")

✅ 图像审核功能已定义


## 3. 视频内容审核功能

In [3]:
def analyze_video_content(video_path: str, custom_prompt: str = None) -> Dict:
    """
    使用Bedrock Nova模型分析视频内容（涉黄涉暴审核）
    
    Args:
        video_path: 视频文件路径
        custom_prompt: 自定义提示词
    
    Returns:
        包含分析结果和时延信息的字典
    """
    start_time = time.time()
    
    try:
        # 读取视频文件并编码为Base64
        with open(video_path, "rb") as video_file:
            binary_data = video_file.read()
            base_64_encoded_data = base64.b64encode(binary_data)
            base64_string = base_64_encoded_data.decode("utf-8")
        
        # 默认审核提示词
        if custom_prompt is None:
            custom_prompt = """请仔细分析这个视频，并以JSON格式返回以下信息（使用中文回答）：
{
  "description": "视频的详细描述",
  "scene_analysis": "场景分析",
  "character_actions": "人物动作描述",
  "contains_violence": "是否包含暴力内容（是/否）",
  "violence_details": "如果包含暴力内容，请详细说明",
  "contains_porn": "是否包含色情内容（是/否）",
  "porn_details": "如果包含色情内容，请详细说明",
  "safety_rating": "安全等级（安全/警告/危险）",
  "confidence_score": "判断的置信度（1-10分）",
  "age_rating": "适合年龄段"
}"""
        
        # 定义系统提示
        system_list = [
            {
                "text": "你是一个专业的视频内容审核专家，擅长识别视频中的暴力和色情内容。请客观、准确地分析视频内容。"
            }
        ]
        
        # 定义用户消息
        message_list = [
            {
                "role": "user",
                "content": [
                    {
                        "video": {
                            "format": "mp4",
                            "source": {"bytes": base64_string},
                        }
                    },
                    {
                        "text": custom_prompt
                    },
                ],
            }
        ]
        
        # 配置推理参数
        inf_params = {"maxTokens": 600, "topP": 0.1, "topK": 20, "temperature": 0.2}
        
        native_request = {
            "schemaVersion": "messages-v1",
            "messages": message_list,
            "system": system_list,
            "inferenceConfig": inf_params,
        }
        
        # 调用Bedrock API
        api_start_time = time.time()
        response = bedrock_client.invoke_model(modelId=MODEL_ID, body=json.dumps(native_request))
        model_response = json.loads(response["body"].read())
        api_end_time = time.time()
        
        # 计算时延
        total_latency = api_end_time - start_time
        api_latency = api_end_time - api_start_time
        
        # 提取响应内容
        content_text = model_response["output"]["message"]["content"][0]["text"]
        
        # 尝试解析JSON响应
        try:
            parsed_content = json.loads(content_text)
        except json.JSONDecodeError:
            # 如果不是有效JSON，创建结构化响应
            parsed_content = {
                "description": content_text,
                "contains_violence": "未知",
                "contains_porn": "未知",
                "safety_rating": "未知",
                "raw_response": content_text
            }
        
        return {
            'success': True,
            'file_path': video_path,
            'file_type': 'video',
            'analysis_result': parsed_content,
            'raw_response': content_text,
            'model_id': MODEL_ID,
            'usage': model_response.get('usage', {}),
            'latency': {
                'total_ms': round(total_latency * 1000, 2),
                'api_ms': round(api_latency * 1000, 2)
            },
            'timestamp': datetime.now().isoformat()
        }
        
    except Exception as e:
        return {
            'success': False,
            'file_path': video_path,
            'file_type': 'video',
            'error': str(e),
            'latency': {
                'total_ms': round((time.time() - start_time) * 1000, 2),
                'api_ms': 0
            },
            'timestamp': datetime.now().isoformat()
        }

print("✅ 视频审核功能已定义")

✅ 视频审核功能已定义


## 4. 统一分析和结果显示功能

In [4]:
def analyze_media_file(file_path: str, custom_prompt: str = None) -> Dict:
    """
    统一的媒体文件分析：自动识别文件类型并调用相应的分析函数
    
    Args:
        file_path: 媒体文件路径
        custom_prompt: 自定义提示词
    
    Returns:
        包含分析结果的字典
    """
    if not os.path.exists(file_path):
        return {'success': False, 'error': f'文件不存在: {file_path}'}
    
    file_ext = os.path.splitext(file_path)[1].lower()
    
    # 图像文件处理
    if file_ext in ['.jpg', '.jpeg', '.png', '.bmp', '.gif']:
        print(f"🖼️ 使用Bedrock Nova分析图像: {os.path.basename(file_path)}")
        return analyze_image_content(file_path, custom_prompt)
    
    # 视频文件处理
    elif file_ext in ['.mp4', '.mov', '.avi']:
        print(f"🎬 使用Bedrock Nova分析视频: {os.path.basename(file_path)}")
        return analyze_video_content(file_path, custom_prompt)
    
    else:
        return {
            'success': False,
            'error': f'不支持的文件格式: {file_ext}',
            'supported_formats': {
                'images': ['.jpg', '.jpeg', '.png', '.bmp', '.gif'],
                'videos': ['.mp4', '.mov', '.avi']
            }
        }

def print_analysis_results(results: Dict):
    """
    格式化打印分析结果
    
    Args:
        results: 分析结果字典
    """
    print("\n" + "="*60)
    print("📊 Bedrock Nova 内容审核结果")
    print("="*60)
    
    if not results.get('success', True):
        print(f"❌ 分析失败: {results.get('error', '未知错误')}")
        return
    
    file_type = results.get('file_type', 'unknown')
    file_path = results.get('file_path', '')
    latency = results.get('latency', {})
    
    print(f"📁 文件: {os.path.basename(file_path)}")
    print(f"📂 类型: {file_type.upper()}")
    print(f"🤖 模型: {results.get('model_id', 'Unknown')}")
    print(f"⏱️ 总延迟: {latency.get('total_ms', 0):.2f}ms")
    print(f"🔗 API延迟: {latency.get('api_ms', 0):.2f}ms")
    
    # 打印分析结果
    analysis = results.get('analysis_result', {})
    if isinstance(analysis, dict):
        print(f"\n🔍 内容审核结果:")
        
        # 基本信息
        if 'description' in analysis:
            print(f"  📝 描述: {analysis['description']}")
        
        if 'scene_analysis' in analysis:
            print(f"  🎬 场景分析: {analysis['scene_analysis']}")
        
        if 'character_actions' in analysis:
            print(f"  👥 人物动作: {analysis['character_actions']}")
        
        # 安全性检查
        violence = analysis.get('contains_violence', '未知')
        porn = analysis.get('contains_porn', '未知')
        safety = analysis.get('safety_rating', '未知')
        confidence = analysis.get('confidence_score', '未知')
        age_rating = analysis.get('age_rating', '未知')
        
        print(f"\n🛡️ 安全性评估:")
        
        # 暴力内容
        if violence == '是':
            print(f"  ⚠️ 暴力内容: {violence}")
            if 'violence_details' in analysis:
                print(f"    详情: {analysis['violence_details']}")
        else:
            print(f"  ✅ 暴力内容: {violence}")
        
        # 色情内容
        if porn == '是':
            print(f"  🔞 色情内容: {porn}")
            if 'porn_details' in analysis:
                print(f"    详情: {analysis['porn_details']}")
        else:
            print(f"  ✅ 色情内容: {porn}")
        
        print(f"  🏷️ 安全等级: {safety}")
        print(f"  📊 置信度: {confidence}")
        print(f"  👶 适合年龄: {age_rating}")
    
    else:
        print(f"\n📝 原始响应: {results.get('raw_response', '')}")
    
    # 打印使用统计
    usage = results.get('usage', {})
    if usage:
        print(f"\n📊 Token使用统计:")
        print(f"  • 输入Token: {usage.get('inputTokens', 0)}")
        print(f"  • 输出Token: {usage.get('outputTokens', 0)}")
        print(f"  • 总Token: {usage.get('totalTokens', 0)}")

print("✅ 统一分析和显示功能已定义")

✅ 统一分析和显示功能已定义


## 5. 使用示例

In [ ]:
# 示例1: 分析图像文件
image_file = "./sample_image.jpg"  # 请替换为实际的图像文件路径

if os.path.exists(image_file):
    print("🖼️ 开始分析图像文件...")
    image_results = analyze_media_file(image_file)
    print_analysis_results(image_results)
else:
    print(f"❌ 图像文件不存在: {image_file}")
    print("请将图像文件放在当前目录下，或修改文件路径")

In [ ]:
# 示例2: 分析视频文件
video_file = "./result.mp4"  # 请替换为实际的视频文件路径

if os.path.exists(video_file):
    print("🎬 开始分析视频文件...")
    video_results = analyze_media_file(video_file)
    print_analysis_results(video_results)
else:
    print(f"❌ 视频文件不存在: {video_file}")
    print("请将视频文件放在当前目录下，或修改文件路径")

In [6]:
# 示例3: 按目录批量处理
def get_media_files_from_directory(directory_path: str, recursive: bool = True) -> List[str]:
    """
    从目录中获取所有支持的媒体文件
    
    Args:
        directory_path: 目录路径
        recursive: 是否递归搜索子目录
    
    Returns:
        媒体文件路径列表
    """
    supported_extensions = {
        '.jpg', '.jpeg', '.png', '.bmp', '.gif',  # 图像格式
        '.mp4', '.mov', '.avi'                    # 视频格式
    }
    
    media_files = []
    
    if not os.path.exists(directory_path):
        print(f"❌ 目录不存在: {directory_path}")
        return media_files
    
    if not os.path.isdir(directory_path):
        print(f"❌ 路径不是目录: {directory_path}")
        return media_files
    
    print(f"📂 扫描目录: {directory_path}")
    print(f"🔍 递归搜索: {'是' if recursive else '否'}")
    
    if recursive:
        # 递归搜索所有子目录
        for root, dirs, files in os.walk(directory_path):
            for file in files:
                file_ext = os.path.splitext(file)[1].lower()
                if file_ext in supported_extensions:
                    full_path = os.path.join(root, file)
                    media_files.append(full_path)
                    print(f"  ✅ 找到文件: {os.path.relpath(full_path, directory_path)}")
    else:
        # 只搜索当前目录
        for file in os.listdir(directory_path):
            file_path = os.path.join(directory_path, file)
            if os.path.isfile(file_path):
                file_ext = os.path.splitext(file)[1].lower()
                if file_ext in supported_extensions:
                    media_files.append(file_path)
                    print(f"  ✅ 找到文件: {file}")
    
    print(f"📊 总共找到 {len(media_files)} 个媒体文件")
    return sorted(media_files)

def batch_analyze_directory(directory_path: str, recursive: bool = True, 
                          max_files: int = None, custom_prompt: str = None):
    """
    批量分析目录中的所有媒体文件
    
    Args:
        directory_path: 目录路径
        recursive: 是否递归搜索子目录
        max_files: 最大处理文件数（None表示处理所有文件）
        custom_prompt: 自定义提示词
    
    Returns:
        分析结果列表
    """
    # 获取目录中的所有媒体文件
    media_files = get_media_files_from_directory(directory_path, recursive)
    
    if not media_files:
        print("❌ 目录中没有找到支持的媒体文件")
        return []
    
    # 限制处理文件数量
    if max_files and len(media_files) > max_files:
        print(f"⚠️ 文件数量超过限制，只处理前 {max_files} 个文件")
        media_files = media_files[:max_files]
    
    results = []
    total_start_time = time.time()
    
    print(f"\n🚀 开始批量分析 {len(media_files)} 个文件...\n")
    
    for i, file_path in enumerate(media_files, 1):
        relative_path = os.path.relpath(file_path, directory_path)
        print(f"📁 处理文件 {i}/{len(media_files)}: {relative_path}")
        
        try:
            result = analyze_media_file(file_path, custom_prompt)
            results.append(result)
            print_analysis_results(result)
        except Exception as e:
            print(f"❌ 处理文件失败: {str(e)}")
            results.append({
                'success': False,
                'file_path': file_path,
                'error': str(e)
            })
        
        print("\n" + "-"*60 + "\n")
        
        # 避免API限流
        time.sleep(1)
    
    total_time = time.time() - total_start_time
    
    # 打印批量处理统计
    successful = [r for r in results if r.get('success', True)]
    failed = [r for r in results if not r.get('success', True)]
    
    print("📊 目录批量处理统计:")
    print(f"  • 处理目录: {directory_path}")
    print(f"  • 总文件数: {len(media_files)}")
    print(f"  • 成功处理: {len(successful)}")
    print(f"  • 处理失败: {len(failed)}")
    print(f"  • 总耗时: {total_time:.2f}秒")
    
    if successful:
        avg_latency = sum(r.get('latency', {}).get('total_ms', 0) for r in successful) / len(successful)
        print(f"  • 平均延迟: {avg_latency:.2f}ms")
        
        # 统计文件类型
        image_count = sum(1 for r in successful if r.get('file_type') == 'image')
        video_count = sum(1 for r in successful if r.get('file_type') == 'video')
        print(f"  • 图像文件: {image_count} 个")
        print(f"  • 视频文件: {video_count} 个")
        
        # 统计安全性结果
        violence_count = sum(1 for r in successful 
                           if r.get('analysis_result', {}).get('contains_violence') == '是')
        porn_count = sum(1 for r in successful 
                        if r.get('analysis_result', {}).get('contains_porn') == '是')
        
        print(f"  • 检测到暴力内容: {violence_count} 个文件")
        print(f"  • 检测到色情内容: {porn_count} 个文件")
        
        # 统计安全等级
        safe_count = sum(1 for r in successful 
                        if r.get('analysis_result', {}).get('safety_rating') == '安全')
        warning_count = sum(1 for r in successful 
                           if r.get('analysis_result', {}).get('safety_rating') == '警告')
        danger_count = sum(1 for r in successful 
                          if r.get('analysis_result', {}).get('safety_rating') == '危险')
        
        print(f"  • 安全等级统计:")
        print(f"    - 安全: {safe_count} 个")
        print(f"    - 警告: {warning_count} 个")
        print(f"    - 危险: {danger_count} 个")
    
    if failed:
        print(f"\n❌ 处理失败的文件:")
        for r in failed:
            print(f"  • {os.path.basename(r.get('file_path', ''))} - {r.get('error', '未知错误')}")
    
    return results

# 使用示例：分析当前目录
current_directory = "./image_input/"  # 当前目录
print("📂 示例：分析当前目录中的媒体文件")

# 方式1: 只分析当前目录（不递归）
print("\n=== 方式1: 分析当前目录（不递归） ===")
batch_results_current = batch_analyze_directory(
    directory_path=current_directory,
    recursive=False,
    max_files=5  # 限制最多处理5个文件
)

# 方式2: 递归分析指定目录及其子目录
target_directory = "./video_input/"  # 请替换为实际的媒体文件目录
print(f"\n=== 方式2: 递归分析目录 {target_directory} ===")

if os.path.exists(target_directory):
    batch_results_recursive = batch_analyze_directory(
        directory_path=target_directory,
        recursive=True,
        max_files=10  # 限制最多处理10个文件
    )
else:
    print(f"❌ 目录不存在: {target_directory}")
    print("请创建目录并放入媒体文件，或修改目录路径")
    batch_results_recursive = []

📂 示例：分析当前目录中的媒体文件

=== 方式1: 分析当前目录（不递归） ===
📂 扫描目录: ./image_input/
🔍 递归搜索: 否
  ✅ 找到文件: cosplay05.png
  ✅ 找到文件: cosplay07.png
📊 总共找到 2 个媒体文件

🚀 开始批量分析 2 个文件...

📁 处理文件 1/2: cosplay05.png
🖼️ 使用Bedrock Nova分析图像: cosplay05.png

📊 Bedrock Nova 内容审核结果
📁 文件: cosplay05.png
📂 类型: IMAGE
🤖 模型: us.amazon.nova-lite-v1:0
⏱️ 总延迟: 5369.29ms
🔗 API延迟: 5363.52ms

🔍 内容审核结果:
  📝 描述: 图像中的女性穿着一套蓝色和黑色的装束，包括一件紧身胸衣和长袖衣物，她的头上有一个羽毛装饰。她坐在一个雕刻有复杂图案的灰色王座上，双腿交叉，穿着黑色高跟鞋和网袜。背景是一个简单的灰色墙壁。

🛡️ 安全性评估:
  ✅ 暴力内容: 否
  🔞 色情内容: 是
    详情: 图像中的女性穿着性感的服装，包括紧身胸衣和网袜，这些元素通常被认为是性感或性暗示的象征。此外，她的姿势和服装的设计也可能被视为性暗示。
  🏷️ 安全等级: 警告
  📊 置信度: 8
  👶 适合年龄: 18岁以上

📊 Token使用统计:
  • 输入Token: 2070
  • 输出Token: 269
  • 总Token: 2339

------------------------------------------------------------

📁 处理文件 2/2: cosplay07.png
🖼️ 使用Bedrock Nova分析图像: cosplay07.png

📊 Bedrock Nova 内容审核结果
📁 文件: cosplay07.png
📂 类型: IMAGE
🤖 模型: us.amazon.nova-lite-v1:0
⏱️ 总延迟: 3253.97ms
🔗 API延迟: 3249.30ms

🔍 内容审核结果:
  📝 描述: ```json
{
  "description": "图像中的女性穿着黑色的蕾丝和亮面材质的连衣裙，头上有一

In [ ]:
# 示例4: 保存分析结果
def save_analysis_results(results: List[Dict], output_file: str = None):
    """
    保存分析结果到JSON文件
    """
    if output_file is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_file = f"bedrock_nova_analysis_results_{timestamp}.json"
    
    try:
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        print(f"✅ 分析结果已保存到: {output_file}")
        return output_file
    except Exception as e:
        print(f"❌ 保存失败: {str(e)}")
        return None

# 保存批量分析结果
all_batch_results = []

# 合并所有批量分析结果
if 'batch_results_current' in locals() and batch_results_current:
    all_batch_results.extend(batch_results_current)
    print(f"✅ 当前目录分析结果: {len(batch_results_current)} 个文件")

if 'batch_results_recursive' in locals() and batch_results_recursive:
    all_batch_results.extend(batch_results_recursive)
    print(f"✅ 递归目录分析结果: {len(batch_results_recursive)} 个文件")

# 保存合并后的结果
if all_batch_results:
    saved_file = save_analysis_results(all_batch_results)
    if saved_file:
        print(f"📄 结果文件大小: {os.path.getsize(saved_file)} 字节")
        print(f"📊 总共保存了 {len(all_batch_results)} 个文件的分析结果")
else:
    print("ℹ️ 没有批量分析结果需要保存")